In [8]:
import pandas as pd
import requests
import geopandas as gpd

pd.set_option("display.max_columns", None)

In [102]:
import pandas as pd

df = pd.read_csv(
    "data/Raw/Unfallorte2024_LinRef.csv",
    sep=";",
    engine="python",
    dtype=str
)

# Remove tabs and extra spaces
df = df.replace("\t", "", regex=True)
df = df.apply(lambda col: col.str.strip())


rename_dict = {
    "OID_": "record_id",
    "UIDENTSTLAE": "accident_id",
    "ULAND": "state_code",
    "UREGBEZ": "administrative_region",
    "UKREIS": "district_code",
    "UGEMEINDE": "municipality_code",
    "UJAHR": "year",
    "UMONAT": "month",
    "USTUNDE": "hour",
    "UWOCHENTAG": "weekday",
    "UKATEGORIE": "accident_severity",
    "UART": "accident_type",
    "UTYP1": "detailed_accident_type",
    "ULICHTVERH": "light_condition",
    "IstStrassenzustand": "road_surface_condition",
    "IstRad": "involves_bicycle",
    "IstPKW": "involves_car",
    "IstFuss": "involves_pedestrian",
    "IstKrad": "involves_motorcycle",
    "IstGkfz": "involves_heavy_vehicle",
    "IstSonstige": "involves_other_vehicle",
    "LINREFX": "road_reference_x",
    "LINREFY": "road_reference_y",
    "XGCSWGS84": "longitude",
    "YGCSWGS84": "latitude",
    "PLST": "location_internal_code"
}

df = df.rename(columns=rename_dict)


In [ ]:
decimal_cols = [
    "road_reference_x",
    "road_reference_y",
    "longitude",
    "latitude"
]

for col in decimal_cols:
    df[col] = (
        df[col].str.replace(",", ".", regex=False)
        .astype(float)
    )

In [104]:
int_cols = [
    "year", "month", "hour", "weekday",
    "accident_severity", "accident_type",
    "detailed_accident_type", "light_condition",
    "road_surface_condition",
    "involves_bicycle", "involves_car",
    "involves_pedestrian", "involves_motorcycle",
    "involves_heavy_vehicle", "involves_other_vehicle"
]

for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 5. Create datetime column
df["datetime"] = pd.to_datetime(
    df["year"].astype(str) + "-" +
    df["month"].astype(str) + "-01"
) + pd.to_timedelta(df["hour"], unit="h")

In [105]:
df.to_csv("data/processed_accidents_2024.csv", index=False)

In [4]:
df=pd.read_csv('data/Processed/processed_accidents_2024.csv')

C:\Users\Ranjith Panicker\AppData\Local\Temp\ipykernel_20736\1296232851.py:1: DtypeWarning: Columns (0: accident_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('data/Processed/processed_accidents_2024.csv')


In [6]:
df[df['involves_bicycle']==1]

,record_id,accident_id,state_code,administrative_region,district_code,municipality_code,year,month,hour,weekday,accident_severity,accident_type,detailed_accident_type,light_condition,road_surface_condition,involves_bicycle,involves_car,involves_pedestrian,involves_motorcycle,involves_heavy_vehicle,involves_other_vehicle,road_reference_x,road_reference_y,longitude,latitude,location_internal_code,datetime
3,4,1240525171013602024,1,0,60,47,2024,5,16,7,3,5,3,0,0,1,1,0,0,0,0,567578.579100,5.963042e+06,10.026341,53.811536,1,2024-05-01 16:00:00
6,7,1240525152013982024,1,0,3,0,2024,5,1,7,3,0,1,2,0,1,0,0,0,0,0,610636.656300,5.968447e+06,10.681975,53.852717,1,2024-05-01 01:00:00
7,8,1240525152013802024,1,0,3,0,2024,5,8,7,3,5,3,0,0,1,1,0,0,0,0,610368.460100,5.971731e+06,10.679081,53.882279,1,2024-05-01 08:00:00
9,10,1240525145013902024,1,0,4,0,2024,5,7,7,3,5,2,0,0,1,1,0,0,0,0,563677.878000,5.991857e+06,9.973117,54.070978,1,2024-05-01 07:00:00
11,12,1240525125013782024,1,0,59,161,2024,5,16,7,3,9,1,0,0,1,0,0,0,0,0,544477.582200,6.064863e+06,9.690664,54.729076,1,2024-05-01 16:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268514,268515,10240927009432524520,10,0,42,116,2024,9,17,6,2,0,7,0,1,1,0,0,0,0,0,351516.032640,5.494759e+06,6.945610,49.587166,2,2024-09-01 17:00:00
268515,268516,10240927009522524460,10,0,45,114,2024,9,15,6,3,2,6,0,0,1,0,0,0,0,0,379086.205409,5.464820e+06,7.336015,49.324126,2,2024-09-01 15:00:00
268516,268517,10241001009422500650,10,0,42,114,2024,10,14,3,2,7,1,0,0,1,0,0,0,0,0,320899.135468,5.486169e+06,6.526286,49.501681,2,2024-10-01 14:00:00
268517,268518,10241011009422509150,10,0,42,115,2024,10,13,6,3,5,3,0,0,1,1,0,0,0,0,318284.435265,5.481027e+06,6.492570,49.454702,2,2024-10-01 13:00:00


In [108]:

wfs_url = "https://www.infravelo.de/api/v1/projects/collections/projekte/"

layers = requests.get(wfs_url).json()
# print(layers)

In [109]:
layers.keys()

dict_keys(['type', 'name', 'features'])

In [110]:
layers['name']

'StandortPin'

In [111]:
d2=pd.DataFrame(layers['features']).drop(columns=['properties'])


In [113]:
d2.to_json("data/Processed/infravelo.json", orient='records')

In [65]:
# d2.to_csv("data/infravelo_projects.csv", index=False)

In [119]:
gdf = gpd.read_file("data/Raw/Long-distance cycling routes-2025.json")
gdf = gdf.to_crs(epsg=4326)
# print(gdf.head())
# print(gdf.crs)

In [121]:
gdf.to_file("data/Processed/Long-distance cycling routes-2025.json", driver='GeoJSON')

In [125]:
gdf = gpd.read_file("data/Raw/Bicycle traffic network-2025.json")
gdf = gdf.to_crs(epsg=4326)
# print(gdf.head())
# print(gdf.crs)

In [127]:
gdf.to_file("data/Processed/Bicycle traffic network-2025.json", driver='GeoJSON')

In [128]:
gdf = gpd.read_file("data/Raw/bicycle streets-2024.json")
gdf = gdf.to_crs(epsg=4326)
# print(gdf.head())
# print(gdf.crs)

In [130]:
gdf.to_file("data/Processed/bicycle streets-2024.json", driver='GeoJSON')

In [131]:
gdf = gpd.read_file("data/Raw/analysis zones-2025.json")
gdf = gdf.to_crs(epsg=4326)
# print(gdf.head())
# print(gdf.crs)

In [133]:
gdf.to_file("data/Processed/analysis zones-2025.json", driver='GeoJSON')

In [134]:
gdf = gpd.read_file("data/Raw/Bicycle infrastructure-2020.json")
gdf = gdf.to_crs(epsg=4326)
# print(gdf.head())
# print(gdf.crs)

In [135]:
gdf

,id,importid,sobj_kz,segm_segm,segm_bez,stst_str,stor_name,ortstl,rva_typ,sorvt_typ,laenge,b_pflicht,geometry
0,b_radverkehrsanlagen.1,1,10-000415,58530025_58530021.01,B1/B5,Alt-Kaulsdorf,Marzahn-Hellersdorf,Kaulsdorf,Radwege,"Geh-/Radweg, durch Markierung unterschieden",26,ja,"MULTILINESTRING ((13.58199 52.50505, 13.58237 ..."
1,b_radverkehrsanlagen.2,2,10-000039,60530006_60530009.01,B1/B5,Alt-Mahlsdorf,Marzahn-Hellersdorf,Mahlsdorf,Radwege,Radfahrerfurt Z 340,16,ja,"MULTILINESTRING ((13.6063 52.50471, 13.60639 5..."
2,b_radverkehrsanlagen.3,3,10-000038,60530005_60530006.01,B1/B5,Alt-Mahlsdorf,Marzahn-Hellersdorf,Mahlsdorf,Radwege,"Geh-/Radweg, durch Markierung unterschieden",510,nein,"MULTILINESTRING ((13.59882 52.50483, 13.59905 ..."
3,b_radverkehrsanlagen.4,4,10-000011,57530025_57530026.01,B1/B5,Alt-Biesdorf,Marzahn-Hellersdorf,Biesdorf,Radwege,"Geh-/Radweg, baulich unterschieden",76,ja,"MULTILINESTRING ((13.56179 52.50846, 13.56214 ..."
4,b_radverkehrsanlagen.5,5,10-000012,57530001_57530017.01,B1/B5,Alt-Biesdorf,Marzahn-Hellersdorf,Biesdorf,Radwege,"Radweg, baulich getrennt",192,ja,"MULTILINESTRING ((13.56277 52.50814, 13.56333 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18636,b_radverkehrsanlagen.18637,18637,04-002513,40560008_40550045.01,NaN,Lise-Meitner-Straße,Charlottenburg-Wilmersdorf,Charlottenburg,Radwege,Radfahrerfurt Z 340,24,ja,"MULTILINESTRING ((13.30663 52.52874, 13.30659 ..."
18637,b_radverkehrsanlagen.18638,18638,04-002514,40560014_40560008.01,NaN,Lise-Meitner-Straße,Charlottenburg-Wilmersdorf,Charlottenburg,Radfahrstreifen,"Radfahrstreifen Z 295, ohne ruh.Verkehr",331,ja,"MULTILINESTRING ((13.30657 52.52899, 13.30642 ..."
18638,b_radverkehrsanlagen.18639,18639,04-002515,40560014_40560008.01,NaN,Lise-Meitner-Straße,Charlottenburg-Wilmersdorf,Charlottenburg,Radfahrstreifen,"Radfahrstreifen Z 295, ohne ruh.Verkehr",136,ja,"MULTILINESTRING ((13.30593 52.53172, 13.30606 ..."
18639,b_radverkehrsanlagen.18640,18640,04-002516,40560014_40560008.01,NaN,Lise-Meitner-Straße,Charlottenburg-Wilmersdorf,Charlottenburg,Schutzstreifen,"Schutzstreifen Z 340, mit ruh.Verkehr mit Begr...",128,ja,"MULTILINESTRING ((13.30624 52.53052, 13.30623 ..."


In [3]:
data=pd.read_csv('data/Raw/dft-road-casualty-statistics-vehicle-last-5-years.csv',nrows=1000)

In [5]:
data.columns

Index(['collision_index', 'collision_year', 'collision_ref_no',
       'vehicle_reference', 'vehicle_type', 'towing_and_articulation',
       'vehicle_manoeuvre_historic', 'vehicle_manoeuvre',
       'vehicle_direction_from', 'vehicle_direction_to',
       'vehicle_location_restricted_lane_historic',
       'vehicle_location_restricted_lane', 'junction_location',
       'skidding_and_overturning', 'hit_object_in_carriageway',
       'vehicle_leaving_carriageway', 'hit_object_off_carriageway',
       'first_point_of_impact', 'vehicle_left_hand_drive',
       'journey_purpose_of_driver_historic', 'journey_purpose_of_driver',
       'sex_of_driver', 'age_of_driver', 'age_band_of_driver',
       'engine_capacity_cc', 'propulsion_code', 'age_of_vehicle',
       'generic_make_model', 'driver_imd_decile', 'lsoa_of_driver',
       'escooter_flag', 'driver_distance_banding'],
      dtype='str')

In [6]:
data.head()

,collision_index,collision_year,collision_ref_no,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre_historic,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane_historic,vehicle_location_restricted_lane,junction_location,skidding_and_overturning,hit_object_in_carriageway,vehicle_leaving_carriageway,hit_object_off_carriageway,first_point_of_impact,vehicle_left_hand_drive,journey_purpose_of_driver_historic,journey_purpose_of_driver,sex_of_driver,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding
0,2020210979534,2020,210979534,2,5,0,3,3,4,6,0,0,0,0,0,0,0,2,1,5,6,1,57,9,798,1,4,BMW F 800,3,E01010422,0,-1
1,2020030959800,2020,030959800,1,5,0,17,19,2,6,0,0,0,0,0,1,0,1,1,6,6,1,59,9,798,1,3,BMW F 800,6,E01025351,0,-1
2,2020360978858,2020,360978858,2,5,0,18,19,8,4,0,0,0,5,0,0,0,1,1,2,2,1,64,9,798,1,11,BMW F 800,4,E01026874,0,-1
3,2020430340711,2020,430340711,1,5,0,18,19,3,7,0,0,0,0,0,7,4,1,1,6,6,1,48,8,798,1,8,BMW F 800,8,E01017758,0,-1
4,2020010260297,2020,010260297,1,5,0,99,99,9,9,99,99,0,9,99,9,99,9,1,2,2,1,36,7,798,1,9,BMW F 800,8,E01016121,0,-1


In [7]:
df=pd.read_csv("data/Processed/processed_accidents_2024.csv")

C:\Users\Ranjith Panicker\AppData\Local\Temp\ipykernel_12688\2429910850.py:1: DtypeWarning: Columns (0: accident_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("data/Processed/processed_accidents_2024.csv")


In [19]:
df.head().to_csv('temp2.csv',index=False)

In [12]:
map=gpd.read_file("data/Processed/Bicycle traffic network-2025.json")

In [18]:
map.head().to_file('temp1.json')

In [21]:
data=pd.read_csv("data/Processed/processed_accidents_2024.csv")

C:\Users\Ranjith Panicker\AppData\Local\Temp\ipykernel_12688\3882689469.py:1: DtypeWarning: Columns (0: accident_id) have mixed types. Specify dtype option on import or set low_memory=False.
  data=pd.read_csv("data/Processed/processed_accidents_2024.csv")


In [24]:
data['datetime']=data['datetime'].astype('datetime64[ns]')

In [26]:
data[['datetime','latitude','longitude']].to_csv('temp3.csv',index=False)

In [ ]:
df=gpd.read_file("data/Processed/Bicycle traffic network-2025.json")

In [32]:
set1=df.sample(40)

In [36]:
import geopandas as gpd
import numpy as np
from datetime import datetime, timedelta
import json


points_per_line = 30
start = datetime(2024,1,1)

features = []

for idx, row in set1.iterrows():

    line = row.geometry
    distances = np.linspace(0, line.length, points_per_line)

    coords = [list(line.interpolate(d).coords)[0] for d in distances]

    timestamps = [
        (start + timedelta(seconds=i*5)).isoformat()
        for i in range(points_per_line)
    ]

    features.append({
        "type":"Feature",
        "properties":{
            "trip_id": idx,
            "timestamps": timestamps
        },
        "geometry":{
            "type":"LineString",
            "coordinates": coords
        }
    })

geojson = {"type":"FeatureCollection","features":features}

with open("animated_trips.geojson","w") as f:
    json.dump(geojson,f)

In [37]:
import json

def transform_to_trips(input_file, output_file):
    with open(input_file, 'r') as f:
        data = json.load(f)

    new_features = []
    
    for feature in data['features']:
        geom = feature['geometry']
        props = feature['properties']
        
        # Function to add time to a list of coordinates
        def add_time(coords):
            timestamp = 1710000000 # Base time
            animated_coords = []
            for pt in coords:
                # Add 2 seconds per point to create movement
                animated_coords.append([pt[0], pt[1], 0, timestamp])
                timestamp += 2
            return animated_coords

        if geom['type'] == 'LineString':
            geom['coordinates'] = add_time(geom['coordinates'])
            new_features.append(feature)
            
        elif geom['type'] == 'MultiLineString':
            # Explode MultiLineString into multiple LineString features
            for line in geom['coordinates']:
                new_feat = {
                    "type": "Feature",
                    "properties": props,
                    "geometry": {
                        "type": "LineString",
                        "coordinates": add_time(line)
                    }
                }
                new_features.append(new_feat)

    new_geojson = {
        "type": "FeatureCollection",
        "features": new_features
    }

    with open(output_file, 'w') as f:
        json.dump(new_geojson, f)

transform_to_trips('data/Processed/Bicycle traffic network-2025.json', 'berlin_trips.json')

In [46]:
import osmnx as ox
import json

# 1. Download Berlin Street Network (Major Roads for speed, or "all" for detail)
# Use "drive" for car roads or "all" for every single alleyway
print("Downloading Berlin street data...")
G = ox.graph_from_place("Berlin, Germany", network_type="drive")

# 2. Convert to GeoDataFrames
nodes, edges = ox.graph_to_gdfs(G)

# Alexanderplatz Center
C_LON, C_LAT = 13.4094, 52.5208
# Your working Unix base
BASE_TIME = 1564184363 
# A "Freeze" time set very far in the future
FOREVER = BASE_TIME + 36000 

features = []

print("Processing radial animation logic...")
for _, row in edges.iterrows():
    coords = list(row['geometry'].coords)
    
    # Calculate radial delay based on the first point of the street
    dist = ((coords[0][0] - C_LON)**2 + (coords[0][1] - C_LAT)**2)**0.5
    # Speed control: Change 25000 to make the expansion faster or slower
    start_ts = int(BASE_TIME + (dist * 25000))
    
    animated_line = []
    for i, pt in enumerate(coords):
        # [Lon, Lat, Alt, Time]
        animated_line.append([pt[0], pt[1], 0, start_ts + (i * 2)])
    
    # Add the "Hold" point to keep the trip visible until the end
    last_pt = animated_line[-1]
    animated_line.append([last_pt[0], last_pt[1], 0, FOREVER])
    
    features.append({
        "type": "Feature",
        "properties": {"highway": row.get('highway', 'street')},
        "geometry": { "type": "LineString", "coordinates": animated_line }
    })

# 3. Save as the file you will upload to Kepler.gl
output = {"type": "FeatureCollection", "features": features}
with open('berlin_real_streets_growing.json', 'w') as f:
    json.dump(output, f)

print("Done! Upload 'berlin_real_streets_growing.json' to Kepler.gl")

Processing radial animation logic...
Done! Upload 'berlin_real_streets_growing.json' to Kepler.gl


In [2]:
import json

with open("Bank_Marketing.ipynb", "r", encoding="utf-8") as f:
    nb = json.load(f)

markdown_text = ""
code_text = ""

for cell in nb["cells"]:
    if cell["cell_type"] == "markdown":
        markdown_text += " ".join(cell["source"]) + " "
    elif cell["cell_type"] == "code":
        code_text += " ".join(cell["source"]) + " "

print("Markdown words:", len(markdown_text.split()))
print("Code words:", len(code_text.split()))
print("Total:", len(markdown_text.split()) + len(code_text.split()))

Markdown words: 2511
Code words: 524
Total: 3035
